# Day 2 — Step 2: 전처리 [필수 과제 — 문제1]

01_data_collection에서 저장한 데이터를 불러와  
가격 관련 파생 지표 3가지를 계산합니다.

---

## 이번 단계에서 만들 컬럼 3가지

| 컬럼 | 계산식 | 의미 |
|------|--------|------|
| `price_change` | `close - open` | 종가 - 시가 → 하루 동안 얼마나 변했나 (원 단위) |
| `price_change_pct` | `price_change / open × 100` | 시가 대비 변동률 (%) |
| `high_low_diff` | `high - low` | 고가 - 저가 → 하루 안에서 얼마나 크게 움직였나 |

---

### price_change가 양수면? 음수면?

```
시가(open) 100만 원 → 종가(close) 105만 원
price_change = 105만 - 100만 = +5만 원  → 양수(+) = 상승 마감

시가(open) 100만 원 → 종가(close)  95만 원  
price_change =  95만 - 100만 = -5만 원  → 음수(-) = 하락 마감
```

### 왜 금액(price_change)이 있는데 변동률(price_change_pct)도 계산하나요?

```
비트코인 +200만 원  vs  솔라나 +200만 원

같은 금액이 올랐지만 비율이 다릅니다:
  BTC : 9,000만 원 기준 +200만 → +2.2%
  SOL :   200만 원 기준 +200만 → +100%

→ 비율(%)로 비교해야 공정한 비교가 됩니다
```

### high_low_diff는 왜 필요한가요?

> 하루 안에서 가격이 얼마나 크게 움직였는지 보여줍니다.  
> 이 값이 클수록 그날 변동성(위험)이 높았다는 의미입니다.

---
## 0. 라이브러리 불러오기

In [ ]:
import pandas as pd     # 데이터를 표(DataFrame)로 다루기
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

---
## 1. 데이터 불러오기

01_data_collection에서 저장한 `ohlcv_raw.csv`를 불러옵니다.

In [ ]:
# pd.read_csv()       : CSV 파일을 DataFrame으로 읽기
# parse_dates=['date']: 'date' 컬럼을 문자열이 아닌 datetime 타입으로 읽기
#   → datetime 타입이어야 나중에 날짜 계산이나 차트 x축에 쓸 수 있음
df = pd.read_csv('ohlcv_raw.csv', parse_dates=['date'])

print(f"데이터 로드 완료: {len(df)}행")
print(df.head())

---
## 2. [필수 과제 — 문제1] 종가와 시가의 차이 계산

In [ ]:
def add_price_features(df):
    """
    가격 관련 파생 컬럼 3개를 계산하여 DataFrame에 추가합니다.

    추가되는 컬럼:
        price_change     : 종가 - 시가
        price_change_pct : 시가 대비 변동률 (%)
        high_low_diff    : 고가 - 저가 (일중 변동폭)

    매개변수(Parameter):
        df : open, high, low, close 컬럼이 있는 DataFrame

    반환값(Return):
        파생 컬럼이 추가된 DataFrame
    """
    # .copy() : 원본 df를 건드리지 않도록 복사본에 작업
    df = df.copy()

    # ── 컬럼1: price_change (종가 - 시가) ────────────────────────
    # DataFrame 컬럼끼리 계산하면 같은 행(row)끼리 자동으로 맞춰서 계산됩니다
    # 즉, df['close'][0] - df['open'][0], df['close'][1] - df['open'][1], ... 을
    # 한 줄로 한 번에 계산할 수 있음 (이걸 벡터 연산이라고 부릅니다)
    df['price_change'] = df['close'] - df['open']

    # ── 컬럼2: price_change_pct (시가 대비 변동률 %) ─────────────
    # (종가 - 시가) / 시가 × 100 = 시가 대비 몇 % 올랐는지
    # price_change를 이미 계산해뒀으니 재사용합니다
    # 예) price_change = +2,000,000, open = 90,000,000
    #     → 2,000,000 / 90,000,000 × 100 ≈ +2.22%
    df['price_change_pct'] = (df['price_change'] / df['open']) * 100

    # ── 컬럼3: high_low_diff (고가 - 저가) ───────────────────────
    # 하루 안에서 가격이 최대 얼마나 움직였는지를 나타냅니다
    df['high_low_diff'] = df['high'] - df['low']

    return df

In [ ]:
# ── 함수 적용 ──────────────────────────────────────────────────────
df = add_price_features(df)

# 새로 추가된 컬럼 확인
# to_string(index=False) : 행 번호 없이 표처럼 출력
print("[문제1 결과 확인 — price_change, price_change_pct, high_low_diff 컬럼 추가됨]")
print()
check_cols = ['ticker', 'date', 'open', 'close', 'price_change', 'price_change_pct', 'high_low_diff']
print(df[check_cols].head(10).to_string(index=False))

In [ ]:
# ── 종목별 평균으로 계산이 맞는지 확인 ────────────────────────────
# groupby('ticker') : 같은 종목끼리 묶어서 통계 계산
# .mean().round(2)  : 평균 계산 후 소수점 2자리까지
print("[종목별 평균 변동률 및 일중 변동폭]")
print(df.groupby('ticker')[['price_change_pct', 'high_low_diff']].mean().round(2))

In [ ]:
# ── 다음 단계로 전달 ──────────────────────────────────────────────
df.to_csv('ohlcv_preprocessed.csv', index=False)
print("ohlcv_preprocessed.csv 저장 완료 → 03_moving_average.ipynb에서 이어서 작업합니다")